In [22]:
import requests
import numpy as np
deploymentName = "ada"
apiVersion = "2023-05-15"
EMBEDDING_MODEL_API = f"https://aicafe.hcl.com/AICafeService/api/v1/subscription/openai/deployments/{deploymentName}/embeddings?api-version={apiVersion}"
SUBSCRIPTION_KEY = "59f971fd-48fb-48cf-adae-d3e1e584c365"

def get_embedding(texts):
    headers = {
        "Content-Type": "application/json",
        "api-key": SUBSCRIPTION_KEY
    }
    payload = {"input": texts}
    
    response = requests.post(url=EMBEDDING_MODEL_API, headers=headers, json=payload)
    
    if response.status_code != 200:
        raise Exception(f"Embedding API error {response.status_code}: {response.text}")
    
    data = response.json()["data"]
    return [np.array(item["embedding"]) for item in data]

# Test
emb = get_embedding(["Hello world"])
print("Embedding vector length:", len(emb[0]))

Embedding vector length: 1536


In [30]:
import requests

deploymentName = "gpt-4.1"
apiVersion = "2024-12-01-preview"
CHAT_MODEL_API = f"https://aicafe.hcl.com/AICafeService/api/v1/subscription/openai/deployments/{deploymentName}/chat/completions?api-version={apiVersion}"
SUBSCRIPTION_KEY = "59f971fd-48fb-48cf-adae-d3e1e584c365"

HEADERS = {
    "Content-Type": "application/json",
    "api-key": SUBSCRIPTION_KEY
}

# Example prompt
prompt = """
You are a factual AI assistant.
Answer concisely.

Context:
The router logs show multiple devices connecting via Wi-Fi.
Each device is assigned an IP by DHCP.
Dnsmasq updates the hosts file frequently.

Question:
What does the router log say about DHCP assignments?

Answer:
"""

payload = {
    "model": "gpt-4.1",
    "messages": [{"role": "user", "content": prompt}],
    "maxTokens": 150,
    "temperature": 0
}

# Call the Chat API
response = requests.post(CHAT_MODEL_API, headers=HEADERS, json=payload)

if response.status_code == 200:
    answer = response.json()['choices'][0]['message']['content']
    print("Chat API Response:\n")
    print(answer)
else:
    print(f"Chat model error {response.status_code}: {response.text}")

Chat API Response:

The router log records when devices connect to Wi-Fi and are assigned IP addresses by DHCP. It shows which device (by MAC address or hostname) received which IP address, and logs each DHCP lease or renewal. Dnsmasq updates the hosts file to reflect these assignments.


In [32]:
deploymentName = "gpt-4.1"
apiVersion = "2024-12-01-preview"
CHAT_MODEL_API = f"https://aicafe.hcl.com/AICafeService/api/v1/subscription/openai/deployments/{deploymentName}/chat/completions?api-version={apiVersion}"
EMBEDDING_MODEL_API = f"https://aicafe.hcl.com/AICafeService/api/v1/subscription/openai/deployments/ada/embeddings?api-version=2023-05-15"
SUBSCRIPTION_KEY = "59f971fd-48fb-48cf-adae-d3e1e584c365"

HEADERS = {
    "Content-Type": "application/json",
    "api-key": SUBSCRIPTION_KEY
}
import re
import numpy as np
import requests
import faiss
from tqdm import tqdm



def chunk_text(text, max_words=120, overlap=30):

    sentences = re.split(r'(?<=[.!?]) +', text)

    chunks = []
    current_chunk = []
    word_count = 0

    for sentence in sentences:
        words = sentence.split()

        if word_count + len(words) > max_words:
            chunks.append(" ".join(current_chunk))

            current_chunk = current_chunk[-overlap:]
            word_count = len(current_chunk)

        current_chunk.extend(words)
        word_count += len(words)

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks
def get_embeddings(texts):

    payload = {"input": texts, "model": "ADA"}

    response = requests.post(
        EMBEDDING_MODEL_API,
        headers=HEADERS,
        json=payload
    )

    if response.status_code != 200:
        raise Exception(response.text)

    data = response.json()["data"]

    return [np.array(item["embedding"]) for item in data]
def retrieve(query, index, chunks, top_k=3):

    query_emb = get_embeddings([query])[0]

    query_emb = np.array([query_emb]).astype("float32")

    distances, indices = index.search(query_emb, top_k)

    results = [chunks[i] for i in indices[0]]

    return results

def rerank(query, docs):

    scores = []

    query_words = set(query.lower().split())

    for doc in docs:

        overlap = len(query_words.intersection(doc.lower().split()))

        scores.append((overlap, doc))

    scores.sort(reverse=True)

    return [doc for _, doc in scores]

PROMPT_TEMPLATE = """
You are a factual AI assistant.

Answer ONLY using the provided context.

Rules:
1. Do not use outside knowledge
2. If answer is missing say:
   "The answer is not available in the provided context."
3. Keep answers concise.

Context:
{context}

Question:
{question}

Answer:
"""

document_text = """
The router logs show multiple devices connecting via Wi-Fi.
Each device is assigned an IP by DHCP.
Dnsmasq updates the hosts file frequently.
The router performs memory cache drops occasionally.
IPv6 multicast messages are normal network maintenance.
"""

query = "What does the router log say about DHCP assignments?"

# --- Step 1: Chunk the document ---
chunks = chunk_text(document_text, max_words=50, overlap=10)

# --- Step 2: Get embeddings for all chunks ---

embeddings = get_embeddings(chunks)
print("Number of embeddings returned:", len(embeddings))
if not embeddings:
    raise Exception("No embeddings returned. Check API key and payload.")

# --- Step 3: Build FAISS index ---
dim = len(embeddings[0])
index = faiss.IndexFlatL2(dim)
index.add(np.array(embeddings).astype("float32"))

# --- Step 4: Retrieve top chunks relevant to the query ---
top_chunks = retrieve(query, index, chunks, top_k=3)

# --- Step 5: Optional rerank ---
top_chunks = rerank(query, top_chunks)

# --- Step 6: Create prompt ---
context = "\n\n".join(top_chunks)
prompt = PROMPT_TEMPLATE.format(context=context, question=query)

print(prompt)

def generate_answer(context, question):
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    
    payload = {
        "model": "gpt-4.1",
        "messages": [{"role": "user", "content": prompt}],
        "maxTokens": 150,
        "temperature": 0
    }

    response = requests.post(CHAT_MODEL_API, headers=HEADERS, json=payload)
    
    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content']
    else:
        print(f"Chat model error {response.status_code}: {response.text}")
        return "Error generating answer"
    

# --- Step 7: Generate the response ---
answer = generate_answer(context, query)
print("\nRetrieved answer:\n")
print(answer)

Number of embeddings returned: 1

You are a factual AI assistant.

Answer ONLY using the provided context.

Rules:
1. Do not use outside knowledge
2. If answer is missing say:
   "The answer is not available in the provided context."
3. Keep answers concise.

Context:
The router logs show multiple devices connecting via Wi-Fi. Each device is assigned an IP by DHCP. Dnsmasq updates the hosts file frequently. The router performs memory cache drops occasionally. IPv6 multicast messages are normal network maintenance.

The router logs show multiple devices connecting via Wi-Fi. Each device is assigned an IP by DHCP. Dnsmasq updates the hosts file frequently. The router performs memory cache drops occasionally. IPv6 multicast messages are normal network maintenance.

The router logs show multiple devices connecting via Wi-Fi. Each device is assigned an IP by DHCP. Dnsmasq updates the hosts file frequently. The router performs memory cache drops occasionally. IPv6 multicast messages are norm

In [35]:
import streamlit as st
import re
import numpy as np
import requests
import faiss
from tqdm import tqdm
from pypdf import PdfReader

# --- CONFIG ---
deploymentName = "gpt-4.1"
apiVersion = "2024-12-01-preview"
CHAT_MODEL_API = f"https://aicafe.hcl.com/AICafeService/api/v1/subscription/openai/deployments/{deploymentName}/chat/completions?api-version={apiVersion}"
EMBEDDING_MODEL_API = f"https://aicafe.hcl.com/AICafeService/api/v1/subscription/openai/deployments/ada/embeddings?api-version=2023-05-15"

# SUBSCRIPTION_KEY = st.secrets["API_KEY"]
SUBSCRIPTION_KEY = "59f971fd-48fb-48cf-adae-d3e1e584c365"

HEADERS = {
    "Content-Type": "application/json",
    "api-key": SUBSCRIPTION_KEY
}

# --- FUNCTIONS ---

def chunk_text(text, max_words=120, overlap=30):
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks = []
    current_chunk = []
    word_count = 0
    for sentence in sentences:
        words = sentence.split()
        if word_count + len(words) > max_words:
            chunks.append(" ".join(current_chunk))
            current_chunk = current_chunk[-overlap:]
            word_count = len(current_chunk)
        current_chunk.extend(words)
        word_count += len(words)
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

def get_embeddings(texts):
    payload = {"input": texts, "model": "ADA"}
    response = requests.post(EMBEDDING_MODEL_API, headers=HEADERS, json=payload)
    if response.status_code != 200:
        st.error(f"Embedding API error: {response.text}")
        return None
    data = response.json()["data"]
    return [np.array(item["embedding"]) for item in data]

def build_faiss_index(chunks, embeddings):
    dim = len(embeddings[0])
    index = faiss.IndexFlatL2(dim)
    index.add(np.array(embeddings).astype("float32"))
    return index

def retrieve(query, index, chunks, top_k=3):
    query_emb = get_embeddings([query])[0]
    distances, indices = index.search(np.array([query_emb]).astype("float32"), top_k)
    return [chunks[i] for i in indices[0]]

def rerank(query, docs):
    scores = []
    query_words = set(query.lower().split())
    for doc in docs:
        overlap = len(query_words.intersection(doc.lower().split()))
        scores.append((overlap, doc))
    scores.sort(reverse=True)
    return [doc for _, doc in scores]

def generate_answer(context, question):
    prompt = f"""
You are a factual AI assistant.

Answer ONLY using the provided context.

Rules:
1. Do not use outside knowledge
2. If answer is missing say:
   "The answer is not available in the provided context."
3. Keep answers concise.

Context:
{context}

Question:
{question}

Answer:
"""
    payload = {
        "model": "gpt-4.1",
        "messages": [{"role": "user", "content": prompt}],
        "maxTokens": 150,
        "temperature": 0
    }
    response = requests.post(CHAT_MODEL_API, headers=HEADERS, json=payload)
    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content']
    else:
        st.error(f"Chat API error: {response.text}")
        return "Error generating answer"

# --- STREAMLIT UI ---

st.title("Advanced RAG QA Demo")
st.write("Upload a TXT or PDF document and ask questions!")

uploaded_file = st.file_uploader("Upload TXT or PDF", type=["txt", "pdf"])

if uploaded_file:
    if uploaded_file.type == "application/pdf":
        reader = PdfReader(uploaded_file)
        document_text = ""
        for page in reader.pages:
            document_text += page.extract_text()
    else:
        document_text = uploaded_file.read().decode()

    st.success("Document loaded!")

    chunks = chunk_text(document_text, max_words=120, overlap=30)
    st.write(f"Created {len(chunks)} chunks for embedding.")

    embeddings = get_embeddings(chunks)
    if not embeddings:
        st.stop()
    
    index = build_faiss_index(chunks, embeddings)
    st.success("Vector index built!")

    question = st.text_input("Ask a question:")

    if question:
        top_chunks = retrieve(question, index, chunks)
        top_chunks = rerank(question, top_chunks)
        context = "\n\n".join(top_chunks)
        answer = generate_answer(context, question)
        
        st.subheader("Answer")
        st.write(answer)

        with st.expander("Retrieved Context"):
            for i, c in enumerate(top_chunks):
                st.markdown(f"**Chunk {i+1}:** {c}")

2026-03-09 16:30:29.351 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 16:30:29.499 
  command:

    streamlit run c:\Users\91962\Desktop\GEN-AI app\.venv\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-09 16:30:29.515 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 16:30:29.515 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 16:30:29.515 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 16:30:29.515 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 16:30:29.515 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 16:30:29.515 Thread '

In [ ]:
import socket
from datetime import datetime

HOST = "0.0.0.0"   # Listen on all interfaces
PORT = 514         # Syslog UDP port (change to 1514 if needed)
BUFFER_SIZE = 8192

def start_syslog_server():
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.bind((HOST, PORT))

    print(f"UDP Syslog server listening on {HOST}:{PORT}")

    while True:
        data, addr = sock.recvfrom(BUFFER_SIZE)
        message = data.decode(errors="ignore")

        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{timestamp}] {addr[0]}:{addr[1]} -> {message.strip()}")

if __name__ == "__main__":
    start_syslog_server()